In [ ]:
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from traffic_intelligence_system.data.stats import compute_split_stats, stats_to_markdown
from traffic_intelligence_system.data.thumbnails import (
    build_thumbnail_matrix,
    load_thumbnail,
)
from traffic_intelligence_system.data.yolo_io import (
    get_split_dirs,
    match_images_to_labels,
)

In [ ]:
ROOT = Path().cwd().parent
RAW = ROOT / "data" / "raw"
print(ROOT)
print(RAW)

In [ ]:
datasets = {
    "citypersons": {
        "root": RAW / "citypersons",
        "layout": "type-first",
        "splits": ["train", "test", "val"],
        "class_names": ["person"],
    },
    "ua_detrac": {
        "root": RAW / "ua_detrac",
        "layout": "split-first",
        "splits": ["train", "test"],
        "class_names": ["car", "van", "bus", "others"],
    },
}

In [ ]:
all_stats = {}
for dataset_name, config in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"Dataset: {dataset_name}")
    print(f"{'=' * 60}")

    stats_by_split = {}
    for split in config["splits"]:
        images_dir, labels_dir = get_split_dirs(config["root"], split, config["layout"])

        stats = compute_split_stats(images_dir, labels_dir)
        stats_by_split[split] = stats

        print(f"\nSplit: {split}")
        print(f"Images: {stats.n_images}")
        print(f"Label files: {stats.n_label_files}")
        print(f"Matched pairs: {stats.n_matched}")
        print(f"Images without labels: {stats.n_images_without_label}")
        print(f"Labels without images: {stats.n_labels_without_image}")
        print(f"Empty labels: {stats.n_empty_labels}")
        print(f"Invalid lines: {stats.n_invalid_lines}")
        print(f"Instances per class: {stats.instances_per_class}")

    all_stats[dataset_name] = stats_by_split

    print("\nMarkdown report:")
    print(stats_to_markdown(stats_by_split, config["class_names"]))

In [ ]:
ua_train_labels = RAW / "ua_detrac" / "train" / "labels"

empty_labels = []

for path in ua_train_labels.rglob("*.txt"):
    if path.is_file() and not path.read_text(encoding="utf-8").strip():
        empty_labels.append(path)

print("Empty label files:", len(empty_labels))

for path in empty_labels:
    print(path.relative_to(RAW / "ua_detrac"))

In [ ]:
ua_root = RAW / "ua_detrac"

train_result = match_images_to_labels(
    ua_root / "train" / "images", ua_root / "train" / "labels"
)

test_result = match_images_to_labels(
    ua_root / "test" / "images", ua_root / "test" / "labels"
)

train_paths = [image for image, _ in train_result.pairs]
test_paths = [image for image, _ in test_result.pairs]

random.seed(0)
sample = random.sample(train_paths, min(30, len(train_paths)))

for path in sample:
    print(path.name)

In [ ]:
def classify_filename(filename):
    if re.match(r"^MVI_\d+_", filename):
        return "a: sequence ID included"
    elif re.match(r"^(?:img|frame)\d+_", filename):
        return "b: frame ID only"
    return "c: other pattern"


pattern_counts = Counter(classify_filename(path.name) for path in train_paths)

for pattern, count in pattern_counts.items():
    print(f"Pattern: {pattern} -> Count: {count}")

In [ ]:
5 & 3

In [ ]:
def get_sequence_id(path):
    match = re.match(r"^(MVI_\d+)_frame\d+", path.name)
    return match.group(1) if match else None


train_sequences = {
    seq for path in train_paths if (seq := get_sequence_id(path)) is not None
}

test_sequences = {
    seq for path in test_paths if (seq := get_sequence_id(path)) is not None
}

shared_sequences = train_sequences & train_sequences

print("Train images:", len(train_paths))
print("Train video sequences:", len(train_sequences))
print("Test images:", len(test_paths))
print("Test video sequences:", len(test_sequences))
print("Shared sequences:", len(shared_sequences))
print("Shared sequence IDs:", sorted(shared_sequences))

In [ ]:
train_paths = sorted(train_paths, key=lambda p: str(p))
test_paths = sorted(test_paths, key=lambda p: str(p))

interim_dir = ROOT / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)

thumbnail_cache = interim_dir / "thumbs_ua_train.npy"

if thumbnail_cache.exists():
    train_matrix = np.load(thumbnail_cache)
    print("Loaded cached thumbnails.")
else:
    train_matrix = build_thumbnail_matrix(train_paths, size=(32, 18))
    np.save(thumbnail_cache, train_matrix)
    print("Built and saved thumbnails.")

print("Matrix shape:", train_matrix.shape)
print("Matrix dtype:", train_matrix.dtype)
print("Expected shape:", (len(train_paths), 32 * 18))

In [ ]:
sample_test_paths = test_paths[:8]

results = []

for test_path in sample_test_paths:
    query = load_thumbnail(test_path, size=(32, 18))

    distances = np.linalg.norm(train_matrix - query, axis=1)

    nearest_indices = np.argsort(distances)[:3]

    results.append(
        {
            "test_path": test_path,
            "nearest_paths": [train_paths[i] for i in nearest_indices],
            "distances": distances[nearest_indices],
        }
    )

print("Number of test images checked:", len(results))
print("Nearest-neighbor search completed.")

In [ ]:
fig, axes = plt.subplots(len(results), 4, figsize=(16, 4 * len(results)))

for row, result in enumerate(results):
    axes[row, 0].imshow(Image.open(result["test_path"]))
    axes[row, 0].set_title("TEST")
    axes[row, 0].axis("off")

    for col, (path, distance) in enumerate(
        zip(result["nearest_paths"], result["distances"]),
        start=1,
    ):
        axes[row, col].imshow(Image.open(path))
        axes[row, col].set_title(f"TRAIN | distance={distance:.3f}")
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
for i, result in enumerate(results):
    print(f"\n--- Test sample {i + 1} ---")
    print("TEST:", result["test_path"].name)

    for path, distance in zip(
        result["nearest_paths"],
        result["distances"],
    ):
        print(f"TRAIN: {path.name} | distance={distance:.3f}")

In [ ]:
print("Number of train images:", len(train_paths))
print("Number of unique train paths:", len(set(train_paths)))

print("Matrix shape:", train_matrix.shape)
print("Number of unique thumbnail rows:", np.unique(train_matrix, axis=0).shape[0])

print("\nTest thumbnail checks:")
for result in results[:3]:
    query = load_thumbnail(result["test_path"], size=(32, 18))
    print(
        result["test_path"].name,
        "| min:",
        round(float(query.min()), 3),
        "| max:",
        round(float(query.max()), 3),
        "| std:",
        round(float(query.std()), 3),
    )

In [ ]:
import random
from collections import defaultdict

test_by_sequence = defaultdict(list)

for path in test_paths:
    sequence_id = get_sequence_id(path)
    if sequence_id is not None:
        test_by_sequence[sequence_id].append(path)

rng = random.Random(42)
sample_test_paths = [rng.choice(paths) for paths in test_by_sequence.values() if paths]

sample_test_paths = sample_test_paths[:8]

print("Selected test images:", len(sample_test_paths))
print("Selected sequences:", [get_sequence_id(path) for path in sample_test_paths])

In [ ]:
results = []

for test_path in sample_test_paths:
    query = load_thumbnail(test_path, size=(32, 18))

    distances = np.linalg.norm(train_matrix - query, axis=1)
    nearest_indices = np.argsort(distances)[:3]

    results.append(
        {
            "test_path": test_path,
            "nearest_paths": [train_paths[i] for i in nearest_indices],
            "distances": distances[nearest_indices],
        }
    )

fig, axes = plt.subplots(
    len(results),
    4,
    figsize=(16, 4 * len(results)),
)

for row, result in enumerate(results):
    axes[row, 0].imshow(Image.open(result["test_path"]))
    axes[row, 0].set_title(f"TEST\n{result['test_path'].name[:30]}")
    axes[row, 0].axis("off")

    for col, (path, distance) in enumerate(
        zip(result["nearest_paths"], result["distances"]),
        start=1,
    ):
        axes[row, col].imshow(Image.open(path))
        axes[row, col].set_title(f"TRAIN | d={distance:.3f}\n{path.name[:25]}")
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
datasets = {
    "UA-DETRAC train": train_paths,
    "UA-DETRAC test": test_paths,
    "CityPersons train": [
        p
        for p in (RAW / "citypersons" / "images" / "train").rglob("*")
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ],
    "CityPersons val": [
        p
        for p in (RAW / "citypersons" / "images" / "val").rglob("*")
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ],
}

for dataset, paths in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"{dataset}\n")

    if not paths:
        print("No images found.")
        continue

    sample_paths = sorted(paths, key=lambda p: str(p))[:5]

    for path in sample_paths:
        with Image.open(path) as img:
            print(f"{path.name} | size={img.size} | mode={img.mode}")

In [ ]:
for dataset_name, paths in datasets.items():
    dimensions = Counter()
    errors = []

    for path in paths:
        try:
            with Image.open(path) as img:
                dimensions[img.size] += 1
        except Exception as exc:
            errors.append((path, str(exc)))

    print(f"\n{'=' * 60}")
    print(dataset_name)
    print("Total images checked:", sum(dimensions.values()))
    print("Image dimensions:")

    for size, count in dimensions.most_common():
        print(f"  {size}: {count} images")

    print("Unreadable images:", len(errors))